In [1]:
# --- Cell 1: load the data and hold back the last 28 days as a test -------------------
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, lightgbm as lgb
from statsmodels.tsa.holtwinters import ExponentialSmoothing

df = pd.read_parquet("data/processed/daily_flows.parquet")
active = pd.read_parquet("data/processed/product_status.parquet").query("status == 'Active'")["product"]
df = df[df["product"].isin(active)].copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["product", "date"])

# When do customers order? Friday is the weekly holiday in Bangladesh
print((df.groupby(df["date"].dt.day_name())["ordered_units"].sum() / df["ordered_units"].sum() * 100).round(1))

HORIZON = 28                                                   # forecast 4 weeks ahead
test_start = df["date"].max() - pd.Timedelta(days=HORIZON - 1)  # 13 Jul 2023
train = df[df["date"] < test_start]
test = df[df["date"] >= test_start].copy()

date
Friday        0.0
Monday       19.1
Saturday      3.6
Sunday       25.6
Thursday     21.6
Tuesday      15.7
Wednesday    14.4
Name: ordered_units, dtype: float64


In [2]:
# --- Cell 2: Model 1 (moving average) and Model 2 (Holt-Winters), one product at a time ---
def moving_average(history, future_dates):
    """Average of the last 28 days, shaped by each weekday's share of orders (the Excel approach)."""
    by_weekday = history.groupby(history["date"].dt.dayofweek)["ordered_units"].mean()
    weekday_factor = (by_weekday / by_weekday.mean()).reindex(range(7), fill_value=1)
    return history["ordered_units"].tail(28).mean() * weekday_factor[future_dates.dt.dayofweek].values

def holt_winters(history, days):
    """Exponential smoothing with a trend and a 7-day (weekday) season."""
    model = ExponentialSmoothing(history["ordered_units"].values, trend="add", damped_trend=True,
                                 seasonal="add", seasonal_periods=7).fit()
    return np.clip(model.forecast(days), 0, None)              # no negative forecasts

for product, history in train.groupby("product"):
    rows = test["product"] == product
    test.loc[rows, "fc_moving_avg"] = moving_average(history, test.loc[rows, "date"])
    test.loc[rows, "fc_holt_winters"] = holt_winters(history, rows.sum())

In [3]:
# --- Cell 3: Model 3, machine learning (LightGBM) trained on all products together ------
def add_features(data):
    """Inputs the model learns from. Every one is at least 28 days old, so a whole
    4-week forecast can be made at once without peeking at the future."""
    data = data.copy()
    past = data.groupby("product")["ordered_units"]
    for weeks_back in [4, 5, 6, 7]:                            # same weekday, 4 to 7 weeks earlier
        data[f"same_day_{weeks_back}w_ago"] = past.shift(7 * weeks_back)
    data["avg_28_days_before"] = past.transform(lambda s: s.shift(28).rolling(28).mean())
    data["weekday"] = data["date"].dt.dayofweek
    data["product_code"] = data["product"].astype("category")
    return data

FEATURES = ["same_day_4w_ago", "same_day_5w_ago", "same_day_6w_ago", "same_day_7w_ago",
            "avg_28_days_before", "weekday", "product_code"]
data = add_features(df)
learn = data[(data["date"] < test_start) & data["avg_28_days_before"].notna()]

model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, min_child_samples=40,
                          objective="tweedie",                 # suits demand with many zero days
                          random_state=42, verbose=-1)
model.fit(learn[FEATURES], learn["ordered_units"])
test["fc_lightgbm"] = np.clip(model.predict(data.loc[test.index, FEATURES]), 0, None)

# Model 4: blend = average of the moving average and LightGBM
test["fc_blend"] = (test["fc_moving_avg"] + test["fc_lightgbm"]) / 2

In [4]:
# --- Cell 4: which model wins? ----------------------------------------------------------
# WAPE = total absolute error / total actual demand (lower is better)
# Bias = (total forecast - total actual) / total actual (+ = over-forecast, - = under-forecast)
test["week"] = test["date"].dt.to_period("W-WED")              # Thursday-Wednesday weeks: 4 full weeks
weekly = test.groupby(["product", "week"]).sum(numeric_only=True)

scores = {}
for m in ["moving_avg", "holt_winters", "lightgbm", "blend"]:
    actual, forecast = test["ordered_units"], test[f"fc_{m}"]
    scores[m] = {
        "WAPE daily %": (actual - forecast).abs().sum() / actual.sum() * 100,
        "WAPE weekly %": (weekly["ordered_units"] - weekly[f"fc_{m}"]).abs().sum() / weekly["ordered_units"].sum() * 100,
        "Bias %": (forecast.sum() - actual.sum()) / actual.sum() * 100,
    }
scores = pd.DataFrame(scores).T.round(1).rename_axis("model")
print(scores)

scores.to_csv("outputs/model_scores.csv")
test.drop(columns="week").to_csv("outputs/forecast_backtest.csv", index=False)

              WAPE daily %  WAPE weekly %  Bias %
model                                            
moving_avg            44.1           26.2    10.4
holt_winters          45.7           28.2    11.2
lightgbm              50.3           28.2    -1.1
blend                 45.8           25.8     4.7


In [5]:
# --- Cell 5: forecast the next 4 weeks with the blend ---------------------------------
future_dates = pd.date_range(df["date"].max() + pd.Timedelta(days=1), periods=HORIZON)   # 10 Aug - 6 Sep 2023
future = pd.DataFrame([(p, d) for p in df["product"].unique() for d in future_dates], columns=["product", "date"])
future["ordered_units"] = np.nan

# LightGBM: retrain on all history, then predict the future rows
everything = add_features(pd.concat([df, future], ignore_index=True))
known = everything[everything["ordered_units"].notna() & everything["avg_28_days_before"].notna()]
model.fit(known[FEATURES], known["ordered_units"])
future_rows = everything[everything["ordered_units"].isna()]
future["fc_lightgbm"] = np.clip(model.predict(future_rows[FEATURES]), 0, None)

# Moving average from the full history, then blend the two
for product, history in df.groupby("product"):
    rows = future["product"] == product
    future.loc[rows, "fc_moving_avg"] = moving_average(history, future.loc[rows, "date"])
future["forecast_units"] = ((future["fc_moving_avg"] + future["fc_lightgbm"]) / 2).round()
future.loc[future["date"].dt.dayofweek == 4, "forecast_units"] = 0   # Fridays: no orders

future[["product", "date", "forecast_units"]].to_csv("outputs/forecast_next_28_days.csv", index=False)
print(future.groupby(future["date"].dt.to_period("W-WED"))["forecast_units"].sum())

date
2023-08-10/2023-08-16    187146.0
2023-08-17/2023-08-23    185069.0
2023-08-24/2023-08-30    188389.0
2023-08-31/2023-09-06    193502.0
Freq: W-WED, Name: forecast_units, dtype: float64
